# Understanding PyTorch by Building It from Scratch

We implement PyTorch's core concepts (autograd, nn.Module, optimizers) using only NumPy.
This reveals the mechanics behind the magic.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)


## 1. Autograd: A Tensor with Gradient Tracking


In [ ]:
class Tensor:
    """A simple autograd tensor that tracks operations for backprop."""
    
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        self.data = np.array(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = np.zeros_like(self.data) if requires_grad else None
        self._backward = lambda: None
        self._children = set(_children)
        self._op = _op
    
    def __repr__(self):
        return f"Tensor({self.data}, grad={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, requires_grad=True, _children=(self, other), _op='+')
        def _backward():
            if self.requires_grad:
                self.grad += out.grad
            if other.requires_grad:
                other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, requires_grad=True, _children=(self, other), _op='*')
        def _backward():
            if self.requires_grad:
                self.grad += other.data * out.grad
            if other.requires_grad:
                other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, requires_grad=True, _children=(self, other), _op='@')
        def _backward():
            if self.requires_grad:
                self.grad += out.grad @ other.data.T
            if other.requires_grad:
                other.grad += self.data.T @ out.grad
        out._backward = _backward
        return out
    
    def sum(self):
        out = Tensor(self.data.sum(), requires_grad=True, _children=(self,), _op='sum')
        def _backward():
            if self.requires_grad:
                self.grad += np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Tensor(np.maximum(0, self.data), requires_grad=True, _children=(self,), _op='relu')
        def _backward():
            if self.requires_grad:
                self.grad += (self.data > 0).astype(float) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        # Topological sort
        topo = []
        visited = set()
        def build(t):
            if t not in visited:
                visited.add(t)
                for child in t._children:
                    build(child)
                topo.append(t)
        build(self)
        self.grad = np.ones_like(self.data)
        for t in reversed(topo):
            t._backward()

print("Tensor class defined with autograd support!")


## 2. Forward Pass: y = w*x + b


In [ ]:
# Simple forward pass
x = Tensor([2.0, 3.0], requires_grad=False)
w = Tensor([0.5, -0.3], requires_grad=True)
b = Tensor([0.1], requires_grad=True)

# y = sum(w * x) + b
y = (w * x).sum() + b

print(f"x = {x.data}")
print(f"w = {w.data}")
print(f"b = {b.data}")
print(f"y = w*x + b = {y.data}")


## 3. Backward Pass: Computing Gradients via Chain Rule


In [ ]:
# Backpropagation
y.backward()

print(f"dy/dw = x = {w.grad}")  # Should be [2.0, 3.0]
print(f"dy/db = 1 = {b.grad}")  # Should be [1.0]
print()
print("Verification: dy/dw should equal x since y = sum(w*x) + b")
print(f"  x = {x.data} ✓" if np.allclose(w.grad, x.data) else "  MISMATCH")


## 4. Computational Graph Visualization (ASCII)


In [ ]:
def print_graph(tensor, indent=0, prefix=""):
    """Print the computational graph as ASCII tree."""
    name = f"[{tensor._op}]" if tensor._op else f"Tensor({tensor.data})"
    grad_str = f" grad={tensor.grad}" if tensor.grad is not None else ""
    print(f"{prefix}{name} -> data={tensor.data}{grad_str}")
    for i, child in enumerate(tensor._children):
        is_last = (i == len(tensor._children) - 1)
        new_prefix = prefix + ("    " if is_last else "│   ")
        connector = "└── " if is_last else "├── "
        print_graph(child, indent + 1, prefix + connector)

print("Computational Graph for y = sum(w*x) + b:")
print("=" * 50)
print_graph(y)


## 5. Implementing nn.Linear from Scratch


In [ ]:
class Linear:
    """Equivalent to torch.nn.Linear."""
    
    def __init__(self, in_features, out_features):
        # Xavier initialization
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.weight = Tensor(np.random.randn(in_features, out_features) * scale, requires_grad=True)
        self.bias = Tensor(np.zeros(out_features), requires_grad=True)
    
    def __call__(self, x):
        return x @ self.weight + Tensor(np.ones((x.data.shape[0], 1))) * self.bias
    
    def parameters(self):
        return [self.weight, self.bias]

layer = Linear(3, 2)
inp = Tensor(np.random.randn(4, 3))  # batch of 4, 3 features
out = layer(inp)
print(f"Input shape: {inp.data.shape}")
print(f"Weight shape: {layer.weight.data.shape}")
print(f"Output shape: {out.data.shape}")


## 6. ReLU Activation with Gradient


In [ ]:
# Demonstrate ReLU forward and backward
x = Tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
y = x.relu().sum()
y.backward()

print("ReLU Forward:")
print(f"  Input:  {x.data}")
print(f"  Output: {np.maximum(0, x.data)}")
print(f"\nReLU Backward (gradient):")
print(f"  dy/dx:  {x.grad}")
print(f"  (0 where input <= 0, 1 where input > 0)")


## 7. Building a 2-Layer Network (like nn.Module)


In [ ]:
class Module:
    """Base class like torch.nn.Module."""
    def parameters(self):
        params = []
        for attr in self.__dict__.values():
            if isinstance(attr, Linear):
                params.extend(attr.parameters())
        return params
    
    def zero_grad(self):
        for p in self.parameters():
            p.grad = np.zeros_like(p.data)

class TwoLayerNet(Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.fc1 = Linear(input_dim, hidden_dim)
        self.fc2 = Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = x.relu()
        x = self.fc2(x)
        return x

net = TwoLayerNet(2, 8, 1)
print(f"Network parameters: {sum(p.data.size for p in net.parameters())}")
print(f"  fc1.weight: {net.fc1.weight.data.shape}")
print(f"  fc1.bias:   {net.fc1.bias.data.shape}")
print(f"  fc2.weight: {net.fc2.weight.data.shape}")
print(f"  fc2.bias:   {net.fc2.bias.data.shape}")


## 8. SGD Optimizer from Scratch


In [ ]:
class SGD:
    """Stochastic Gradient Descent optimizer."""
    
    def __init__(self, parameters, lr=0.01):
        self.parameters = parameters
        self.lr = lr
    
    def step(self):
        for p in self.parameters:
            if p.grad is not None:
                p.data -= self.lr * p.grad
    
    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

optimizer = SGD(net.parameters(), lr=0.1)
print("SGD Optimizer created with lr=0.1")


## 9. Training on XOR Problem


In [ ]:
# XOR dataset
X_train = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float64)
y_train = np.array([[0],[1],[1],[0]], dtype=np.float64)

# Fresh network
net = TwoLayerNet(2, 16, 1)
optimizer = SGD(net.parameters(), lr=0.5)

# MSE Loss
def mse_loss(pred, target):
    diff = Tensor(pred.data - target, requires_grad=False)
    loss = (pred + Tensor(-target)) 
    # Manual MSE: sum of (pred - target)^2
    err = pred.data - target
    return np.mean(err**2), err

losses = []
for epoch in range(500):
    # Forward
    x_t = Tensor(X_train)
    pred = net.forward(x_t)
    
    # Compute loss (MSE)
    error = pred.data - y_train
    loss_val = np.mean(error**2)
    losses.append(loss_val)
    
    # Backward (manual gradient of MSE)
    optimizer.zero_grad()
    
    # d(MSE)/d(pred) = 2*(pred - target)/n
    grad_output = 2 * error / len(y_train)
    
    # Backprop through fc2
    net.fc2.weight.grad = net.fc1(x_t).relu().data.T @ grad_output
    net.fc2.bias.grad = grad_output.sum(axis=0)
    
    # Backprop through relu
    hidden = net.fc1(x_t).data
    relu_grad = (hidden > 0).astype(float)
    
    # Backprop through fc1
    delta = (grad_output @ net.fc2.weight.data.T) * relu_grad
    net.fc1.weight.grad = X_train.T @ delta
    net.fc1.bias.grad = delta.sum(axis=0)
    
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss_val:.6f}")

print(f"\nFinal predictions:")
final_pred = net.forward(Tensor(X_train)).data
for i in range(4):
    print(f"  {X_train[i]} -> {final_pred[i][0]:.4f} (target: {y_train[i][0]})")


## 10. Loss Curve


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('XOR Training Loss Curve')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. The Same in Real PyTorch

```python
import torch
import torch.nn as nn

# Define network
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# Training
model = XORNet()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
criterion = nn.MSELoss()

X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

for epoch in range(500):
    pred = model(X)
    loss = criterion(pred, y)
    optimizer.zero_grad()
    loss.backward()  # autograd handles everything!
    optimizer.step()
```

**Key difference**: PyTorch's autograd automatically computes all gradients via `loss.backward()`. 
Our manual version needed explicit gradient formulas for each layer.


## 12. Comparison: Our Implementation vs PyTorch


In [ ]:
comparison = '''
┌─────────────────────┬──────────────────────────┬─────────────────────────┐
│ Concept             │ Our Implementation       │ PyTorch Equivalent      │
├─────────────────────┼──────────────────────────┼─────────────────────────┤
│ Tensor + autograd   │ class Tensor (manual)    │ torch.tensor(req_grad)  │
│ Linear layer        │ class Linear             │ nn.Linear               │
│ Activation          │ .relu() method           │ torch.relu / nn.ReLU    │
│ Network             │ class TwoLayerNet        │ nn.Module subclass      │
│ Optimizer           │ class SGD                │ torch.optim.SGD         │
│ Backward pass       │ Manual gradient formulas │ loss.backward()         │
│ Comp. graph         │ _children, _op tracking  │ Built-in DAG            │
└─────────────────────┴──────────────────────────┴─────────────────────────┘
'''
print(comparison)


## 13. DataLoader: Mini-Batch Iteration


In [ ]:
class DataLoader:
    """Mini-batch data loader like torch.utils.data.DataLoader."""
    
    def __init__(self, X, y, batch_size=32, shuffle=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
    
    def __iter__(self):
        n = len(self.X)
        indices = np.arange(n)
        if self.shuffle:
            np.random.shuffle(indices)
        for start in range(0, n, self.batch_size):
            idx = indices[start:start + self.batch_size]
            yield self.X[idx], self.y[idx]
    
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

# Demo with synthetic data
X_big = np.random.randn(100, 2)
y_big = (X_big[:, 0] > X_big[:, 1]).astype(float).reshape(-1, 1)

loader = DataLoader(X_big, y_big, batch_size=16)
print(f"Dataset size: {len(X_big)}")
print(f"Batch size: 16")
print(f"Number of batches: {len(loader)}")
print(f"\nIterating:")
for i, (xb, yb) in enumerate(loader):
    print(f"  Batch {i}: X shape={xb.shape}, y shape={yb.shape}")


## 14. Learning Rate Scheduling: Cosine Annealing


In [ ]:
class CosineAnnealingLR:
    """Cosine annealing learning rate scheduler."""
    
    def __init__(self, optimizer, T_max, eta_min=0):
        self.optimizer = optimizer
        self.T_max = T_max
        self.eta_min = eta_min
        self.base_lr = optimizer.lr
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        new_lr = self.eta_min + (self.base_lr - self.eta_min) *                  (1 + np.cos(np.pi * self.current_step / self.T_max)) / 2
        self.optimizer.lr = new_lr
        return new_lr

# Visualize
optimizer_demo = SGD([], lr=0.1)
scheduler = CosineAnnealingLR(optimizer_demo, T_max=100, eta_min=0.001)

lrs = []
for _ in range(100):
    lrs.append(scheduler.step())

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Cosine Annealing Learning Rate Schedule')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 15. Summary

We built from scratch:
1. **Tensor with autograd** - tracks operations and computes gradients
2. **Linear layer** - matrix multiply + bias
3. **ReLU** - activation with proper gradient
4. **Module** - container for parameters
5. **SGD** - gradient descent optimizer
6. **DataLoader** - mini-batch iteration
7. **LR Scheduler** - cosine annealing

These are the exact same abstractions PyTorch provides. The difference is PyTorch's implementation is in C++/CUDA for performance, handles arbitrary computation graphs automatically, and supports GPU acceleration.
